# Bootstraping

## Actividad

1. Tomar el dataset "Motor Trend Road"

2. Realizar regresión lineal sobre mpg = b0 + b1(hp) + b2(qsec)

   * Calcular intervalos de confianza 95% de los betas

3. Crear 1000 muestras de bootstrap con m observaciones con reemplazo

   * Realizar la regresión mpg = b0 + b1(hp) + b2(qsec)

   * Calcular Bx y sigma_b, abrir intervalos de confianza

4. Comparar resultados entre inciso 2 y 3


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import seaborn as sns

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Motor Trend Car Road Tests.xlsx to Motor Trend Car Road Tests.xlsx


In [3]:
df = pd.read_excel("Motor Trend Car Road Tests.xlsx")
df.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


## 2. Regresión Lineal

In [4]:
X = df[['hp', 'qsec']]
X = sm.add_constant(X)  # beta0

y = df['mpg']

In [5]:
modelo = sm.OLS(y, X).fit()

In [6]:
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     25.43
Date:                Mon, 04 May 2026   Prob (F-statistic):           4.18e-07
Time:                        14:19:06   Log-Likelihood:                -86.170
No. Observations:                  32   AIC:                             178.3
Df Residuals:                      29   BIC:                             182.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         48.3237     11.103      4.352      0.0

### 2.1 IC 95% de los betas

In [7]:
ic = modelo.conf_int(alpha=0.05)
ic.columns = ['Límite inferior', 'Límite superior']
print(ic)

       Límite inferior  Límite superior
const        25.614894        71.032516
hp           -0.113089        -0.056097
qsec         -1.979929         0.206770


## 3. 1000 muestras con Bootstrap

In [8]:
n = len(df)
betas = []

In [9]:
for i in range(1000):
    # muestra con reemplazo
    muestra = df.sample(n=n, replace=True)

    X_b = muestra[['hp', 'qsec']]
    X_b = sm.add_constant(X_b)
    y_b = muestra['mpg']

    modelo_b = sm.OLS(y_b, X_b).fit()

    betas.append(modelo_b.params.values)

betas = np.array(betas)

### 3.2 IC 95% bootstrap

In [10]:
medias = betas.mean(axis=0)
stds = betas.std(axis=0)

print("Medias:", medias)
print("Desviaciones:", stds)

Medias: [50.17578768 -0.08754447 -0.97177326]
Desviaciones: [11.20999776  0.0170538   0.53546431]


In [11]:
ic_df = pd.DataFrame({
    "Límite Inferior": medias - 2*stds,
    "Límite Superior": medias + 2*stds
}, index=["b0", "b1", "b2"])

print(ic_df)

    Límite Inferior  Límite Superior
b0        27.755792        72.595783
b1        -0.121652        -0.053437
b2        -2.042702         0.099155


## 4. Comparación inciso 2 y 3

Al comparar los resultados del método clásico y el bootstrap, se observa que ambos producen intervalos de confianza muy similares para los coeficientes del modelo.

En particular, el coeficiente de hp es significativo en ambos métodos, ya que sus intervalos no incluyen el valor 0. Por otro lado, el coeficiente de qsec no es significativo en ninguno de los dos casos, ya que sus intervalos sí incluyen el 0.

Esto muestra que ambos métodos llegan a las mismas conclusiones, por lo que los resultados del modelo son confiables.

# Aggregating

## Actividad

1. Dividir los datos en train y test (50/50)
2. Generar 1000 modelos de regresión lineal, seleccionando en cada uno 3 variables al azar
3. Entrenar cada modelo con train y predecir sobre test
4. Calcular el R2 de cada modelo
5. Promediar las predicciones de los 1000 modelos para obtener ytest
6. Evaluar el modelo agregado calculando el R2 final

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [13]:
X = df.drop(columns=["mpg", "model"])
y = df["mpg"]

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

In [15]:
columnas = X.columns
resultados = []

In [16]:
for i in range(1000):
    vars_random = np.random.choice(columnas, size=3, replace=False) # 3 variables al azar
    Xi_train = X_train[list(vars_random)]

    modelo = LinearRegression()
    modelo.fit(Xi_train, y_train)

    resultados.append({
        "variables": vars_random,
        "modelo": modelo
    })

In [17]:
predicciones = []

In [18]:
for res in resultados:
    vars_modelo = res["variables"]
    modelo = res["modelo"]

    Xi_test = X_test[list(vars_modelo)]  # datos de test con mismas variables

    y_pred = modelo.predict(Xi_test)
    predicciones.append(y_pred)

In [19]:
predicciones = np.array(predicciones)
predicciones.shape

(1000, 16)

In [20]:
y_pred_promedio = predicciones.mean(axis=0)
y_pred_promedio

array([20.11954142, 10.35682349, 14.4749252 , 27.14207095, 23.51461256,
       20.21606347, 13.46451688, 27.4728174 , 15.23581965, 21.69993419,
       15.44798543, 10.60595001, 19.75585262, 15.19709883, 14.73135819,
       13.59082815])

In [21]:
r2_final = r2_score(y_test, y_pred_promedio)
print("R2 final (bagging):", r2_final)

R2 final (bagging): 0.7878967732226485


El valor de R2 indica que el modelo explica aproximadamente el 78.8% de la variabilidad de la variable mpg en el conjunto de prueba, lo cual representa un buen ajuste.

Esto muestra que el uso de aggregating permite mejorar la estabilidad y el desempeño del modelo, ya que al promediar múltiples modelos se reduce la variabilidad de las predicciones.